In [ ]:
!pip install transformers==4.41.2 sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import pandas as pd

import transformers

from transformers import pipeline

print(transformers.__version__)

4.41.2


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
absa_df = pd.read_csv(
    '/content/drive/MyDrive/absa_results.csv'
)

print(absa_df.head())

                                              review    aspect  \
0  instead of going to a jewelry store to purchas...  bracelet   
1  instead of going to a jewelry store to purchas...   jewelry   
2  sadly had to return as toe area too shallow we...      shoe   
3  sadly had to return as toe area too shallow we...       fit   
4  sadly had to return as toe area too shallow we...      ring   

  aspect_sentiment  
0         POSITIVE  
1         NEGATIVE  
2         POSITIVE  
3         NEGATIVE  
4         NEGATIVE  


In [ ]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    tokenizer="google/flan-t5-base"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
positive_df = absa_df[
    absa_df['aspect_sentiment'] == 'POSITIVE'
]

print(positive_df.head())

                                               review    aspect  \
0   instead of going to a jewelry store to purchas...  bracelet   
2   sadly had to return as toe area too shallow we...      shoe   
5   my husband and i both desperately needed slipp...     color   
7   my husband and i both desperately needed slipp...      size   
11  to my surprise it fit like it was made for me ...       fit   

   aspect_sentiment  
0          POSITIVE  
2          POSITIVE  
5          POSITIVE  
7          POSITIVE  
11         POSITIVE  


In [ ]:
grouped = positive_df.groupby(
    'review'
)['aspect'].apply(list).reset_index()

print(grouped.head())

                                              review           aspect
0  bought as a gift for a relative the panties ap...        [quality]
1  bought them for my mom cause she said she coul...     [heel, size]
2  buy size up from yours bra looks great and fee...           [size]
3  charm part is okay but i loved the chain took ...          [chain]
4  cute heels that fit to size the only thing is ...  [heel, comfort]


In [ ]:
def generate_ad(aspects):

    aspect_text = ", ".join(aspects)

    prompt = f"""
    You are a professional luxury fashion marketing copywriter.

    Create a premium, emotionally engaging,
    stylish ecommerce advertisement for a fashion product.

    Highlight these customer-loved features:
    {aspect_text}

    The advertisement should:
    - sound modern and premium
    - feel emotional and persuasive
    - attract young online shoppers
    - be catchy and professional
    - include fashion marketing language
    - be 2 to 4 lines long

    Advertisement:
    """

    result = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=1.0,
        top_k=50,
        top_p=0.95
    )

    return result[0]['generated_text']

In [ ]:
test_aspects = [
    'fit',
    'comfort',
    'style'
]

print(generate_ad(test_aspects))

There are a range of ecommerce websites out there that offer a variety of fashion items. These may contain a few ecommerce options. The type of product you are advertising is something that's commonly used by shoppers. This can be a fashion product, which you can adorn with leather slippers, short dresses or tops. It can also be a


In [ ]:
generated_ads = []

for index, row in grouped.head(20).iterrows():

    review = row['review']

    aspects = row['aspect']

    ad = generate_ad(aspects)

    generated_ads.append({

        'review': review,

        'positive_aspects': aspects,

        'generated_ad': ad
    })

In [ ]:
ads_df = pd.DataFrame(generated_ads)

In [ ]:
print(ads_df.head(20))

                                               review      positive_aspects  \
0   bought as a gift for a relative the panties ap...             [quality]   
1   bought them for my mom cause she said she coul...          [heel, size]   
2   buy size up from yours bra looks great and fee...                [size]   
3   charm part is okay but i loved the chain took ...               [chain]   
4   cute heels that fit to size the only thing is ...       [heel, comfort]   
5   fits well and looks ok i havent tried them in ...                 [fit]   
6   for the price i was surprised over all the met...               [price]   
7   good fit but i wore it out while i was at a ba...                 [fit]   
8   great cotton large sock it is always hard to f...                [size]   
9   great earrings for the price needed some new s...                [ring]   
10  i am female with a small regular acu top i bou...                 [fit]   
11  i bought the superfeet berry to use in my runn..

In [ ]:
ads_df.to_csv(

    '/content/drive/MyDrive/generated_ads.csv',

    index=False
)

print("ADS GENERATED SUCCESSFULLY")

ADS GENERATED SUCCESSFULLY
